# NVDA BB Mean-Reversion — Paper Trading (延迟行情)
**策略规则：**
- 延迟价格 < BB 下轨 → 买入现金的 10%
- 次日延迟价格仍 < BB 下轨 → 再买入现金的 15%
- 首次建仓后第 5 个交易日收盘前卖出全部

**运行要求：** TWS 开着，Paper Trading 账户，端口 7497

## 1. 参数设置

In [ ]:
# ── 策略参数 ──────────────────────────────────────────
SYMBOL     = 'NVDA'
BB_WINDOW  = 20
BB_STD     = 2.0
HOLD_DAYS  = 5
BUY1_PCT   = 0.10
BUY2_PCT   = 0.15
PORT       = 7497    # Paper Trading
CLIENT_ID  = 10
STATE_FILE = f'state_{SYMBOL}.json'   # 持仓状态存档
# ──────────────────────────────────────────────────────

## 2. IBKR 连接 + 工具函数

In [ ]:
import threading, time, json, os
from datetime import datetime, date
import pandas as pd
import numpy as np
from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract
from ibapi.order import Order


class IBApp(EWrapper, EClient):
    def __init__(self):
        EClient.__init__(self, self)
        self.ready       = threading.Event()
        self.hist_done   = threading.Event()
        self.hist_data   = []
        self.next_oid    = None
        self.last_price  = None
        self.cash        = None
        self.account_done = threading.Event()
        self.order_status = {}

    # ── 连接 ──────────────────────────────────────────
    def nextValidId(self, orderId):
        self.next_oid = orderId
        self.ready.set()
        print(f'[TWS] 连接成功，nextOrderId={orderId}')

    def error(self, reqId, errorCode, errorString, advancedOrderRejectJson=''):
        if errorCode not in (2104, 2106, 2107, 2108, 2158):
            print(f'[ERROR] {errorCode}: {errorString}')

    # ── 历史数据 ──────────────────────────────────────
    def historicalData(self, reqId, bar):
        self.hist_data.append({'date': bar.date, 'close': bar.close})

    def historicalDataEnd(self, reqId, start, end):
        self.hist_done.set()

    # ── 延迟行情 tick ─────────────────────────────────
    def tickPrice(self, reqId, tickType, price, attrib):
        if tickType in (4, 68) and price > 0:   # 4=Last, 68=Delayed Last
            self.last_price = price

    # ── 账户资金 ──────────────────────────────────────
    def updateAccountValue(self, key, val, currency, accountName):
        if key == 'CashBalance' and currency == 'USD':
            try:
                self.cash = float(val)
            except:
                pass

    def accountDownloadEnd(self, accountName):
        self.account_done.set()

    # ── 订单状态 ──────────────────────────────────────
    def orderStatus(self, orderId, status, filled, remaining,
                    avgFillPrice, permId, parentId, lastFillPrice,
                    clientId, whyHeld, mktCapPrice):
        self.order_status[orderId] = status
        print(f'[ORDER] {orderId} → {status}  filled={filled}  avgPrice={avgFillPrice}')


def make_contract(symbol='NVDA'):
    c = Contract()
    c.symbol   = symbol
    c.secType  = 'STK'
    c.exchange = 'SMART'
    c.currency = 'USD'
    return c


def connect_app(port=PORT, client_id=CLIENT_ID):
    app = IBApp()
    app.connect('127.0.0.1', port, clientId=client_id)
    threading.Thread(target=app.run, daemon=True).start()
    app.ready.wait(timeout=10)
    if not app.ready.is_set():
        raise RuntimeError('连接 TWS 失败，请确认 TWS 已开启')
    # 设置延迟行情（免费）
    app.reqMarketDataType(3)
    print('[TWS] 已设置延迟行情模式（15-20分钟延迟）')
    return app

print('函数定义完成')

## 3. 状态管理（持仓记录）

In [ ]:
def load_state():
    """从文件读取持仓状态，程序重启后可恢复"""
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE) as f:
            s = json.load(f)
        print(f'[STATE] 读取到持仓状态: {s}')
        return s
    return {
        'position':    0.0,       # 当前持股数
        'entry_date':  None,      # 首次建仓日期（字符串 YYYY-MM-DD）
        'second_done': False,     # 是否已做第二次加仓
        'avg_cost':    0.0,       # 平均成本
    }

def save_state(state):
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f, indent=2)
    print(f'[STATE] 状态已保存: {state}')

def trading_days_since(entry_date_str):
    """计算从建仓日到今天经过了几个交易日（简单用日历天数估算）"""
    if entry_date_str is None:
        return 0
    entry = datetime.strptime(entry_date_str, '%Y-%m-%d').date()
    today = date.today()
    # 简单计算工作日数
    days = np.busday_count(entry, today)
    return int(days)

state = load_state()
print(f'当前状态: {state}')

## 4. 计算 BB 下轨

In [ ]:
def get_bb_lower(app, symbol=SYMBOL, window=BB_WINDOW, n_std=BB_STD):
    """拉最近 N+10 天历史数据，计算当前 BB 下轨"""
    app.hist_data = []
    app.hist_done.clear()
    app.reqHistoricalData(
        reqId=2, contract=make_contract(symbol), endDateTime='',
        durationStr='2 M', barSizeSetting='1 day',
        whatToShow='TRADES', useRTH=1, formatDate=1,
        keepUpToDate=False, chartOptions=[]
    )
    app.hist_done.wait(timeout=30)
    df = pd.DataFrame(app.hist_data)
    df['close'] = df['close'].astype(float)
    mid   = df['close'].rolling(window).mean().iloc[-1]
    std   = df['close'].rolling(window).std().iloc[-1]
    lower = mid - n_std * std
    upper = mid + n_std * std
    print(f'[BB] Mid={mid:.2f}  Upper={upper:.2f}  Lower={lower:.2f}')
    return lower


def get_delayed_price(app, symbol=SYMBOL, wait=5):
    """订阅延迟行情，等待价格返回"""
    app.last_price = None
    app.reqMktData(reqId=3, contract=make_contract(symbol),
                   genericTickList='', snapshot=False, regulatorySnapshot=False,
                   mktDataOptions=[])
    for _ in range(wait * 10):
        if app.last_price is not None:
            break
        time.sleep(0.1)
    app.cancelMktData(reqId=3)
    print(f'[PRICE] {symbol} 延迟价格 = {app.last_price}')
    return app.last_price


def get_cash(app):
    """查询账户可用现金"""
    app.cash = None
    app.account_done.clear()
    app.reqAccountUpdates(True, '')
    app.account_done.wait(timeout=10)
    app.reqAccountUpdates(False, '')
    print(f'[ACCOUNT] 可用现金 = ${app.cash:,.2f}')
    return app.cash

print('BB/Price/Cash 函数定义完成')

## 5. 下单函数

In [ ]:
def place_order(app, action, quantity, symbol=SYMBOL):
    """下市价单，返回 orderId"""
    if quantity <= 0:
        print(f'[ORDER] 数量为0，跳过')
        return None
    oid = app.next_oid
    app.next_oid += 1

    order = Order()
    order.action        = action          # 'BUY' or 'SELL'
    order.orderType     = 'MKT'
    order.totalQuantity = quantity
    order.tif           = 'DAY'

    app.placeOrder(oid, make_contract(symbol), order)
    print(f'[ORDER] {action} {quantity} shares of {symbol}  orderId={oid}')
    return oid

print('下单函数定义完成')

## 6. 主策略逻辑（每天运行一次）

In [ ]:
def run_daily_check():
    """
    每个交易日运行一次（建议收盘后或盘中均可）。
    逻辑：
      1. 持仓超过 HOLD_DAYS → 卖出
      2. 延迟价格 < BB 下轨 → 买入
    """
    print(f'\n{'='*50}')
    print(f'[{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}] 开始每日检查')

    app = connect_app()
    state = load_state()

    try:
        price     = get_delayed_price(app)
        bb_lower  = get_bb_lower(app)
        cash      = get_cash(app)
        today_str = date.today().strftime('%Y-%m-%d')

        if price is None:
            print('[WARN] 无法获取价格，跳过今日检查')
            return

        days_held = trading_days_since(state['entry_date'])
        print(f'[INFO] price={price:.2f}  bb_lower={bb_lower:.2f}  '
              f'position={state["position"]:.2f}  days_held={days_held}')

        # ── 卖出：持仓满 HOLD_DAYS ──────────────────────
        if state['position'] > 0 and days_held >= HOLD_DAYS:
            qty = int(state['position'])
            print(f'[SELL] 持仓 {days_held} 天，触发卖出 {qty} 股')
            place_order(app, 'SELL', qty)
            state['position']    = 0.0
            state['entry_date']  = None
            state['second_done'] = False
            state['avg_cost']    = 0.0
            save_state(state)
            return

        # ── 买入：价格下穿 BB 下轨 ──────────────────────
        if price < bb_lower:
            # 第一次建仓
            if state['position'] == 0 and cash and cash > 0:
                spend = cash * BUY1_PCT
                qty   = int(spend / price)
                print(f'[BUY_1] 价格 {price:.2f} < BB下轨 {bb_lower:.2f}，买入 {qty} 股（10%）')
                place_order(app, 'BUY', qty)
                state['position']   = float(qty)
                state['entry_date'] = today_str
                state['avg_cost']   = price
                save_state(state)

            # 第二次加仓：次日仍在下轨以下
            elif (state['position'] > 0
                  and not state['second_done']
                  and days_held == 1
                  and cash and cash > 0):
                spend = cash * BUY2_PCT
                qty   = int(spend / price)
                print(f'[BUY_2] 次日仍 < BB下轨，加仓 {qty} 股（15%）')
                place_order(app, 'BUY', qty)
                state['position']    += float(qty)
                state['second_done']  = True
                save_state(state)
        else:
            print(f'[INFO] 价格 {price:.2f} >= BB下轨 {bb_lower:.2f}，无信号')

    finally:
        time.sleep(1)
        app.disconnect()
        print('[TWS] 已断开连接')

print('主逻辑定义完成，运行下一个 cell 开始执行')

## 7. 执行（每天手动运行一次，或设定定时任务）

In [ ]:
# 手动运行：每个交易日盘中或收盘后执行一次
run_daily_check()

## 8. 查看当前持仓状态

In [ ]:
state = load_state()
days  = trading_days_since(state['entry_date'])
print(f"持股数量 : {state['position']}")
print(f"建仓日期 : {state['entry_date']}")
print(f"已持有   : {days} 个交易日（满 {HOLD_DAYS} 天卖出）")
print(f"第二次加仓: {'已完成' if state['second_done'] else '未触发'}")